# Day 058 — Exercise 3: Streaming Chat API

Combine `StreamingResponse` with Ollama's streaming mode to build a chat endpoint that yields tokens as SSE events. We use **dependency injection** via an optional `stream_fn` parameter — pass a fake generator in tests, use real Ollama in production.

In [ ]:
import json
from fastapi import FastAPI
from fastapi.responses import StreamingResponse
from pydantic import BaseModel, Field
from starlette.testclient import TestClient


## Task

Implement `build_streaming_chat_api(stream_fn=None)` — return a FastAPI app with:

```
POST /chat/stream   body: {"prompt": "..."}   → SSE token stream
```

- If `stream_fn` is not None: call `stream_fn(prompt)` → iterate tokens
- If `stream_fn` is None: use `ollama.chat(stream=True)` + `extract_tokens`
- Emit each token as `data: {"token": "..."}\n\n`
- End with `data: [DONE]\n\n`
- `Field(min_length=1)` → Pydantic returns 422 on empty prompt automatically

## Your Implementation

In [ ]:
class ChatRequest(BaseModel):
    prompt: str = Field(min_length=1)


def build_streaming_chat_api(stream_fn=None) -> FastAPI:
    """Build a FastAPI app with POST /chat/stream.

    stream_fn: optional callable(prompt: str) -> Iterable[str]
               Pass a fake generator for testing (no Ollama).
               If None, uses ollama.chat(model='llama3.2', stream=True).

    The endpoint:
    - Accepts JSON body {"prompt": "..."}
    - Returns 422 if prompt is empty (Pydantic Field(min_length=1) does this)
    - Streams SSE: 'data: {"token": "..."}\\n\\n' then 'data: [DONE]\\n\\n'
    """
    # TODO: build app with POST /chat/stream → StreamingResponse
    raise NotImplementedError


In [ ]:
def build_streaming_chat_api(stream_fn=None) -> FastAPI:
    app = FastAPI()

    class _ChatRequest(BaseModel):
        prompt: str = Field(min_length=1)

    @app.post("/chat/stream")
    def chat_stream(req: _ChatRequest):
        def generate():
            if stream_fn is not None:
                tokens = stream_fn(req.prompt)
            else:
                import ollama
                chunks = ollama.chat(
                    model="llama3.2",
                    messages=[{"role": "user", "content": req.prompt}],
                    stream=True,
                )
                tokens = (
                    c["message"]["content"]
                    for c in chunks
                    if c["message"]["content"]
                )
            for token in tokens:
                payload = json.dumps({"token": token})
                yield "data: " + payload + "\n\n"
            yield "data: [DONE]\n\n"

        return StreamingResponse(generate(), media_type="text/event-stream")

    return app


## Automated checks

In [ ]:
score, total = 0, 5

def fake_stream(prompt):
    for token in ["Hello", " ", "world", "!"]:
        yield token

try:
    app    = build_streaming_chat_api(stream_fn=fake_stream)
    client = TestClient(app, raise_server_exceptions=False)

    r = client.post("/chat/stream", json={"prompt": "hi"})
    assert r.status_code == 200, f"Expected 200, got {r.status_code}"
    score += 1; print("\u2705 POST /chat/stream returns 200")

    ct = r.headers.get("content-type", "")
    assert "text/event-stream" in ct, f"Expected text/event-stream, got {ct}"
    score += 1; print("\u2705 content-type is text/event-stream")

    assert "data:" in r.text, "Expected 'data:' SSE lines in body"
    assert '"token"' in r.text, "Expected JSON with 'token' key"
    score += 1; print('\u2705 body contains SSE data: lines with token JSON')

    assert "[DONE]" in r.text, "[DONE] sentinel missing"
    score += 1; print("\u2705 [DONE] sentinel present")

    r2 = client.post("/chat/stream", json={"prompt": ""})
    assert r2.status_code == 422, f"Expected 422 for empty prompt, got {r2.status_code}"
    score += 1; print("\u2705 empty prompt returns 422")

except Exception as e:
    print(f"\u274c {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python
def build_streaming_chat_api(stream_fn=None) -> FastAPI:
    app = FastAPI()

    class _ChatRequest(BaseModel):
        prompt: str = Field(min_length=1)

    @app.post("/chat/stream")
    def chat_stream(req: _ChatRequest):
        def generate():
            if stream_fn is not None:
                tokens = stream_fn(req.prompt)
            else:
                import ollama
                chunks = ollama.chat(
                    model="llama3.2",
                    messages=[{"role": "user", "content": req.prompt}],
                    stream=True,
                )
                tokens = (
                    c["message"]["content"]
                    for c in chunks
                    if c["message"]["content"]
                )
            for token in tokens:
                payload = json.dumps({"token": token})
                yield "data: " + payload + "\n\n"
            yield "data: [DONE]\n\n"

        return StreamingResponse(generate(), media_type="text/event-stream")

    return app
```

**Why it works:** The inner class `_ChatRequest` inherits Pydantic validation including the `min_length=1` constraint — empty prompt → 422 before `generate()` is ever called. The generator dispatches on `stream_fn` for testability.

</details>